# Personal Folder Agent — fine-tuning on Colab Pro+

This notebook runs **stage 2** of the pipeline: it fine-tunes Qwen 2.5 on the
text chunks produced by the crawl / extract / chunk stages, which you run on
your own machine (that is where your folders live — nothing is uploaded except
the chunk file you choose to upload).

## Before you start

1. **Runtime → Change runtime type → A100 GPU**, and tick **High-RAM**.
   The pipeline works on an L4 or a T4 too, but the batch settings it picks
   for an A100 are what make a 3B fine-tune finish in a reasonable time. The
   notebook tells you exactly which GPU you got in step 3.
2. **Runtime → Run in background** (Pro+ only). A background run keeps
   training after you close the tab. Without it, training stops when the
   browser disconnects.
3. Have `chunks.jsonl` on your Google Drive. Produce it locally with:

   ```python
   from personal_folder_agent.utils.chunk_io import save_chunks_jsonl
   save_chunks_jsonl(chunks, "chunks.jsonl")
   ```

Everything is written to Drive, so if the session is recycled mid-run you can
reconnect and continue from the last completed epoch (step 8).

## 1. Check the allocated GPU

`nvidia-smi` is the ground truth. An A100 shows ~40GB or ~80GB; an L4 ~23GB; a
T4 ~15GB. If you asked for an A100 and got a T4, change the runtime type and
re-run — training will otherwise take several times longer.

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

Checkpoints, the exported model and the run metadata all live under Drive so
they survive the session.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2b. Put `personal_folder_agent.zip` and `chunks.jsonl` on Drive

Runs the browser file picker for whichever of the two files is not on Drive
yet, then copies it into `MyDrive/personal-folder-agent/` so a recycled
session never needs a second upload.

In [ ]:
import os
from google.colab import files

DRIVE_DIR = '/content/drive/MyDrive/personal-folder-agent'
os.makedirs(DRIVE_DIR, exist_ok=True)

WANTED = ('personal_folder_agent.zip', 'chunks.jsonl')
missing = [name for name in WANTED if not os.path.exists(os.path.join(DRIVE_DIR, name))]

if missing:
    print('choose:', ', '.join(missing))
    for name, data in files.upload().items():
        with open(os.path.join(DRIVE_DIR, name), 'wb') as handle:
            handle.write(data)
        print('saved to Drive:', name)
else:
    print('both files already on Drive - nothing to upload')


## 3. Install dependencies and put the package on the path

`bitsandbytes` is what makes the 8-bit optimizer available; without it the
pipeline falls back to plain AdamW, which needs roughly 24GB more VRAM for the
optimizer state and will not fit a 3B full fine-tune on a 40GB A100.

Adjust `PROJECT_DIR` to wherever you copied the `personal_folder_agent`
package on Drive.

In [ ]:
!pip install -q "transformers>=4.45" "datasets>=2.20" "accelerate>=0.30" "trl>=0.10" "bitsandbytes>=0.43" "tiktoken>=0.7" "pyyaml>=6.0"

In [ ]:
import os, sys, zipfile

DRIVE_DIR = '/content/drive/MyDrive/personal-folder-agent'
PROJECT_DIR = '/content/pfa'

os.makedirs(PROJECT_DIR, exist_ok=True)
with zipfile.ZipFile(os.path.join(DRIVE_DIR, 'personal_folder_agent.zip')) as archive:
    archive.extractall(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)

import personal_folder_agent
print('package loaded from', personal_folder_agent.__file__)


## 4. Confirm what the pipeline sees

`detect_environment()` reports the accelerator and the hardware tier the batch
settings come from. The Colab tier hint is *inferred from the GPU* — Colab does
not expose your subscription level to the runtime — so treat it as a sanity
check that you got Pro+ hardware, not as an authoritative answer.

In [ ]:
from personal_folder_agent.utils.environment import detect_environment, recommend_settings

environment = detect_environment()
print(environment.describe())
print('colab tier hint:', environment.colab_tier_hint)
print('RAM (GB):', round(environment.total_ram_gb or 0, 1))
for note in environment.detection_notes:
    print('note:', note)

settings = recommend_settings(environment, batch_size=2, gradient_accumulation_steps=4)
print()
print('per-device batch     :', settings.per_device_batch_size)
print('gradient accumulation:', settings.gradient_accumulation_steps)
print('effective batch      :', settings.effective_batch_size, '(same on every GPU)')
print('precision            :', settings.precision)
print('optimizer            :', settings.optim)

## 5. Load the chunks

Point `CHUNKS_FILE` at the `chunks.jsonl` you exported locally.

In [ ]:
from personal_folder_agent.utils.chunk_io import load_chunks_jsonl

CHUNKS_FILE = '/content/drive/MyDrive/personal-folder-agent/chunks.jsonl'  # <- edit me

chunks = load_chunks_jsonl(CHUNKS_FILE)
total_tokens = sum(chunk.token_count for chunk in chunks)
print(f'{len(chunks)} chunks, {total_tokens:,} tokens, '
      f'{len({chunk.source_file for chunk in chunks})} source files')

## 6. Configure the run

Checkpoints go to Drive so a recycled session does not cost you the run. Leave
`auto_tune_hardware` on: it re-splits `batch_size x gradient_accumulation_steps`
for whichever GPU you got while keeping the effective batch size identical, so
the run means the same thing on an A100 as on a T4 — it just finishes sooner.

In [ ]:
from personal_folder_agent.components.training_pipeline import TrainingConfig, QwenTrainingPipeline

config = TrainingConfig(
    model='Qwen2.5-3B-Instruct',
    epochs=3,
    batch_size=2,
    learning_rate=2e-5,
    gradient_accumulation_steps=4,
    max_length=1024,
    checkpoint_dir='/content/drive/MyDrive/personal-folder-agent/checkpoints',
    output_dir='/content/drive/MyDrive/personal-folder-agent/output',
)

pipeline = QwenTrainingPipeline(config, progress_callback=print)


def show(value, digits=4):
    """Round a metric, or say so when it was never measured."""
    return 'not measured' if value is None else round(value, digits)


## 7. Train

The first run downloads the base model (~6GB), which takes a few minutes.
Progress lines report the split, the hardware decisions, and any OOM recovery.

In [ ]:
metadata = pipeline.train(chunks)

summary = pipeline.run_summary
print()
print('final training loss  :', round(metadata.performance_metrics.final_loss, 4))
print('final validation loss:', show(metadata.performance_metrics.final_validation_loss))
print('final accuracy       :', round(metadata.performance_metrics.final_accuracy, 4))
print('duration (s)         :', round(metadata.performance_metrics.training_duration_seconds, 1))
print('peak GPU memory (MB) :', round(metadata.performance_metrics.peak_memory_mb, 1))
print('converged            :', summary.converged)
print('success              :', summary.success)
for note in summary.notes:
    print('note:', note)

## 8. If the session was recycled mid-run

Reconnect, re-run steps 2–6, then run this cell instead of step 7. The chunk
set and hyperparameters were saved next to the checkpoints, so training picks
up from the last completed epoch — you do not need to re-upload anything.

In [ ]:
from personal_folder_agent.components.training_pipeline import find_latest_checkpoint

latest = find_latest_checkpoint(config.checkpoint_dir)
print('resuming from', latest)

metadata = pipeline.resume_from_checkpoint(config.checkpoint_dir)
print('final validation loss:', show(metadata.performance_metrics.final_validation_loss))

## 9. What you end up with

`training_metadata.json` in the output directory records the dataset
statistics, the hyperparameters actually used (including the hardware-resolved
batch settings), the loss curve verdict, and any OOM reductions that happened
along the way.

In [ ]:
import json

with open(f'{config.output_dir}/training_metadata.json', encoding='utf-8') as handle:
    print(json.dumps(json.load(handle), indent=2)[:2000])

## 10. Export the model

Writes `model.pkl`, `model_metadata.json` and a versioned archive
(`personal_agent_v{major}.{minor}.{patch}_{YYYY-MM-DDTHH-MM-SS}.zip`) to the
output directory on Drive. Export refuses to run if training did not complete
or if the crawl produced no files or tokens — a model trained on nothing looks
like a success until you query it.

In [ ]:
from personal_folder_agent.components.model_exporter import ExportConfig, LocalModelExporter

export_config = ExportConfig(
    output_dir=config.output_dir,
    version='1.0.0',
    folder_list=sorted({chunk.source_file.rsplit('/', 1)[0] for chunk in chunks}),
)

exporter = LocalModelExporter(export_config, progress_callback=print)
export_result = exporter.export_model(pipeline.trained_model, metadata)

print()
print('archive :', export_result.archive_name)
print('saved to:', export_result.local_path)
print('success :', export_result.success)

## 11. Optional: upload the archive to Drive

Only run this if you want a copy outside the output directory. Nothing is
uploaded without the confirmation callback returning True, and only the
`drive.file` OAuth scope is requested — this app can see the files it creates
and nothing else in your Drive.

Set `encryption_passphrase` to encrypt the archive with AES-256-GCM before it
leaves the runtime. Keep the passphrase: `decrypt_file()` in
`model_exporter.py` is the only way back.

In [ ]:
# Taken from last_artifacts, not built from output_dir: if the export fell
# back to a temp directory, output_dir does not hold the archive. Read it
# before `exporter` is rebound below.
archive_path = exporter.last_artifacts.archive_path

export_config.drive_enabled = True
export_config.drive_folder_id = None          # None = your Drive root
export_config.encryption_passphrase = None    # set a passphrase to encrypt

exporter = LocalModelExporter(
    export_config,
    confirmation_callback=lambda prompt: input(prompt + ' [y/N] ').strip().lower() == 'y',
    progress_callback=print,
)

upload = exporter.upload_to_google_drive(
    str(archive_path),
    export_config.drive_folder_id or '',
)
print('uploaded :', upload.success)
print('encrypted:', upload.encrypted)
print('url      :', upload.drive_url or upload.error_message)